# Q-Learning CIA Test 1 — FrozenLake Debugging
**Marks:15**
## Instructions

The following program implements a **Q-learning delivery robot using a custom 4 × 4 FrozenLake environment**.

- Start state = 3
- Goal state = 12
- Correct the errors without completely rewriting the program.
- Execute the corrected program and verify the learned route and final reward.

### Environment

```text
F F F S
F F H F
F F F F
G F F F
```
State numbering is row-wise from 0 to 15.

In [1]:
# Q-Learning using a custom FrozenLake environment

import numpy as np
import random
import gymnasium as gym

custom_map = [
    "FFFS",
    "FFHF",
    "FFFF",
    "GFFF"
]

env = gym.make(
    "FrozenLake-v1",
    desc=custom_map,
    is_slippery=True
)

n_states = env.observation_space.n
n_actions = env.action_space.n

Q_table = np.ones((n_states, n_actions))

learning_rate = 0.01
discount_factor = 0.50
epsilon = 0.2
episodes = 100

# 0 = Left, 1 = Down, 2 = Right, 3 = Up

def choose_action(state):
    #Here basically > is the bug and if we changed it to < it will give us proper output because random.random if greater than epsilon not 
    # and one more thing it is using np.argmin else it should use argmax
    if random.random() < epsilon:
        return random.randrange(n_actions)
    else:
        return np.argmax(Q_table[state])

for episode in range(episodes):

    state, info = env.reset()

    for step in range(100):

        action = choose_action(state)

        next_state, reward, terminated, truncated, info = env.step(action)
        # so here bug is that it is written AND which basically means that if both happens then only it will be done but it should be either terminated or truncated
        done = terminated or truncated
        #so here np minn should be changed to np max 
        best_next_q = np.min(Q_table[next_state])
        #so here it should be instead of learning rate discount rate 
        target = reward + learning_rate * best_next_q

        if done:
            target = reward
        
        Q_table[next_state, action] = (
            Q_table[state, action]
            + learning_rate *
            (target - Q_table[state, action])
        )

        state = next_state

        if done:
            break

print("Learned Q-table:")
print(np.round(Q_table, 3))

action_symbols = {
    0: "←",
    1: "↓",
    2: "→",
    3: "↑"
}

grid = env.unwrapped.desc

print("\nLearned Delivery Robot Policy")
print("-----------------------------")

for row in range(4):

    line = ""

    for col in range(4):

        state = row * 4 + col
        cell = grid[row][col].decode()

        if cell == "S":
            symbol = "S"
        elif cell == "G":
            symbol = "G"
        elif cell == "H":
            symbol = "X"
        else:
            best_action = np.argmin(Q_table[state])
            symbol = action_symbols[best_action]

        line += f"[ {symbol} ] "

    print(line)

state, info = env.reset()

start_state = state
path = [state]
total_reward = 0

for step in range(30):

    action = np.argmin(Q_table[state])

    next_state, reward, terminated, truncated, info = env.step(action)

    total_reward += reward
    path.append(next_state)

    state = next_state

    if terminated and truncated:
        break

print("\nStart state:", start_state)
print("Path followed:", path)
print("Total reward:", total_reward)

goal_state = 15

if state == goal_state:
    print("Delivery successful!")
else:
    print("Delivery unsuccessful.")

env.close()


ModuleNotFoundError: No module named 'gymnasium'